# Technical Architecture & Data Engineering Pipeline

## 1. Infrastructure Strategy
Standard business intelligence tools (like Power BI or Excel) face severe memory constraints and UI rendering latency when directly ingesting datasets exceeding 20 million rows. Attempting to run Exploratory Data Analysis (EDA) on the raw 3.5GB Parquet file using Pandas would result in out-of-memory crashes.

To bypass these limitations, this project utilized a two-tiered architectural approach:
1. **Exploratory Phase (Pandas + DuckDB):** Extracted a lightweight 5% random sample (approx. 1.01 million rows) using DuckDB to prototype cleaning logic and statistical thresholds in Pandas.
2. **Production Phase (DuckDB SQL):** Executed a custom, in-memory pipeline using DuckDB to clean and pre-aggregate the entire 22-million row dataset down to a 185,000-row semantic model in roughly 16 seconds.

## 2. Exploratory Data Analysis & Rule Definition
The raw TLC High Volume For-Hire Vehicle dataset contains significant physical and financial anomalies. Using the 5% sample, I developed the following programmatic cleaning rules based on statistical realities rather than hardcoded assumptions.

### Chronological and Physical Anomalies
Initial queries revealed rows with impossible physics and corrupted timestamps. The following baseline rules were established:
* **Chronological Flow:** Request time must precede scene arrival, which must precede pickup, which must precede drop-off.
* **Teleports & Ghosts:** Dropped any trips with `trip_miles <= 0` or negative base fares.
* **The 50 MPH Proxy:** Calculated average speed (`trip_miles / (trip_time / 3600)`). While highway speeds reach 65 mph, an average speed exceeding 50 mph across an entire trip within the NYC grid is physically anomalous and indicates GPS distortion. Trips exceeding this average were dropped.

### The "Inverted Pay" Elbow Method
A significant anomaly was discovered where drivers earned more than the passenger paid (`driver_pay > base_passenger_fare`). Initial counts showed this applied to over 160,000 trips in the sample, indicating platform bonuses rather than strict data errors. 

To determine the anomalous threshold, I calculated a `pay_ratio` and tested multiple thresholds (1.25x, 1.5x, 1.75x, 2x). Plotting these results revealed a sharp drop-off and flattening immediately after the 1.5x mark. 
* **The Rule:** Any trip where driver pay exceeded 1.5x the base passenger fare was classified as an extreme anomaly and removed, which successfully trimmed the long tail (representing roughly 1.33% of the dataset).

### Distribution Skew & The "Hidden Ingredients"
Visualizing the distribution of base fares versus trip miles revealed a heavy right-skew. The mileage distribution dropped off significantly faster than the base fare distribution. This proved that distance alone is a poor predictor of fare cost, as "hidden ingredients" (tolls, time delays, and surge multipliers) heavily pad the fares.
* **The Rule:** Because of this skew, using a mean average or hardcoded caps to trim outliers would compromise the data. Instead, the pipeline uses DuckDB's `quantile_cont` function to dynamically trim the 99th percentile.


## 3. The Production Pipeline
The final deployment script applies the rules discovered during the EDA phase to the full 22-million row dataset. 

Because airport runs (JFK/LaGuardia) inherently distort time and distance metrics, the 99th percentile thresholds are calculated and applied independently for 'Airport' trips (where `airport_fee > 0`) and standard 'City' trips. The cleaned data is then instantly aggregated along dimensional axes (Date, Hour, Zone, Company) to feed the Power BI semantic model.

### Final DuckDB Deployment Script


In [ ]:
import duckdb
import time

# Raw TLC HVFHV dataset (22 Million Rows)
raw_file_path = 'hvfhv_full_dataset.parquet' 
output_file_master = 'Master_Dashboard.csv'

print("Processing 22 million rows in-memory via DuckDB...")
start_time = time.time()

unified_pipeline_query = f"""
COPY (
    -- STEP 1: Calculate Segmented Outlier Thresholds (Dynamic 99th Percentile)
    WITH Thresholds AS (
        SELECT 
            CASE WHEN airport_fee > 0 THEN 'Airport' ELSE 'City' END AS trip_segment,
            quantile_cont(base_passenger_fare, 0.99) AS p99_fare,
            quantile_cont(trip_miles, 0.99) AS p99_miles,
            quantile_cont(trip_time, 0.99) AS p99_time,
            quantile_cont(driver_pay, 0.99) AS p99_driver_pay
        FROM '{raw_file_path}'
        GROUP BY CASE WHEN airport_fee > 0 THEN 'Airport' ELSE 'City' END
    ),
    
    -- STEP 2: Apply Physical, Chronological, and "Elbow Method" Constraints
    CleanedBase AS (
        SELECT *,
               CASE WHEN airport_fee > 0 THEN 'Airport' ELSE 'City' END AS trip_segment
        FROM '{raw_file_path}'
        WHERE request_datetime <= on_scene_datetime            
          AND on_scene_datetime <= pickup_datetime             
          AND pickup_datetime < dropoff_datetime               
          AND base_passenger_fare > 0                          
          AND trip_miles > 0                                   
          AND trip_time > 60 AND trip_time < 14400             
          AND (trip_miles / (trip_time / 3600.0)) <= 50        
          AND (driver_pay / base_passenger_fare) <= 1.5        
    ),
    
    -- STEP 3: Trim Top 1% Outliers by Segment
    FullyCleaned AS (
        SELECT c.* EXCLUDE (trip_segment)
        FROM CleanedBase c
        JOIN Thresholds t 
          ON c.trip_segment = t.trip_segment
        WHERE c.base_passenger_fare <= t.p99_fare
          AND c.trip_miles <= t.p99_miles
          AND c.trip_time <= t.p99_time
          AND c.driver_pay <= t.p99_driver_pay
    )
    
    -- STEP 4: Dimensional Pre-Aggregation for Power BI
    SELECT 
        CAST(request_datetime AS DATE) AS request_date,
        DAYNAME(request_datetime) AS day_of_week,
        CASE WHEN DAYNAME(request_datetime) IN ('Saturday', 'Sunday') THEN 'Weekend' 
        ELSE 'Weekday' END AS day_type,
        HOUR(request_datetime) AS hour_of_day,
        PULocationID,
        CASE WHEN hvfhs_license_num = 'HV0003' THEN 'Uber' 
             WHEN hvfhs_license_num = 'HV0005' THEN 'Lyft' 
             ELSE hvfhs_license_num END AS company,
        
        -- Volume & Wait Time Metrics
        COUNT(*) AS trip_volume,
        AVG(date_diff('minute', request_datetime, on_scene_datetime)) AS avg_driver_travel_time,
        AVG(date_diff('minute', on_scene_datetime, pickup_datetime)) AS avg_passenger_boarding_time,
        AVG(date_diff('minute', request_datetime, pickup_datetime)) AS avg_total_wait_time,
        SUM(CASE WHEN shared_request_flag = 'Y' THEN 1 ELSE 0 END) AS total_shared_requests,
        SUM(CASE WHEN shared_match_flag = 'Y' THEN 1 ELSE 0 END) AS total_shared_matches,
        
        -- Financial Metrics
        SUM(base_passenger_fare) AS total_revenue,
        AVG(tips) AS avg_tip_amount,
        SUM(tips) AS total_tips,
        SUM(base_passenger_fare) / NULLIF(SUM(trip_miles), 0) AS revenue_per_mile,
        SUM(driver_pay) / NULLIF((SUM(trip_time) / 3600.0), 0) AS driver_hourly_earnings
        
    FROM FullyCleaned
    WHERE PULocationID IS NOT NULL
      AND PULocationID NOT IN (1, 264, 265) 
    GROUP BY 
        request_date, day_of_week, day_type, hour_of_day, PULocationID, company

) TO '{output_file_master}' (HEADER, DELIMITER ',');
"""

duckdb.sql(unified_pipeline_query)
end_time = time.time()
print(f"Pipeline executed successfully in {round(end_time - start_time, 2)} seconds.")